# ChaDongCha — Vehicle Classifier Training

**EfficientNet-Lite B2** fine-tuned on our vehicle class catalog.

### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your scraped images to Google Drive:
   ```
   MyDrive/chadongcha/images/<ClassName>/0001.jpg ...
   ```
   Run `scrape_images.py` locally first (see ml/training/scrape_images.py)
3. Run all cells top to bottom


In [ ]:
# ── 1. Mount Drive & set paths ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR   = '/content/drive/MyDrive/chadongcha/images'
OUTPUT_DIR = '/content/drive/MyDrive/chadongcha/models'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive mounted. Classes found:', len(os.listdir(DATA_DIR)))

In [ ]:
# ── 2. Install deps ───────────────────────────────────────────────────────────
!pip install -q timm torchmetrics

In [ ]:
# ── 3. Imports ────────────────────────────────────────────────────────────────
import torch, timm, json, shutil
from pathlib import Path
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchmetrics import Accuracy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')

In [ ]:
# ── 4. Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE   = 260      # EfficientNet-Lite B2 native resolution
BATCH_SIZE = 32
EPOCHS     = 40
LR         = 3e-4
VAL_SPLIT  = 0.15
MIN_IMAGES_PER_CLASS = 20   # drop classes with fewer images (too noisy)

# NOTE: _Background class is included in the training data to give the model
# a "none of the above" option. Without it, softmax always forces a winner
# even on non-vehicle inputs (thighs, walls, etc. get high-confidence results).
# The background class should appear in class_map.json — the app checks for it
# and treats it as a classification miss (confidence = 0 effectively).

In [ ]:
# ── 5. Dataset ────────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(DATA_DIR)

# Drop classes with too few images
valid_classes = [
    cls for cls, idx in full_dataset.class_to_idx.items()
    if sum(1 for _, l in full_dataset.samples if l == idx) >= MIN_IMAGES_PER_CLASS
]
print(f'Classes with >= {MIN_IMAGES_PER_CLASS} images: {len(valid_classes)} / {len(full_dataset.classes)}')

# Filter samples to valid classes only
valid_idx_set = {full_dataset.class_to_idx[c] for c in valid_classes}
full_dataset.samples = [(p, l) for p, l in full_dataset.samples if l in valid_idx_set]

# Remap labels to 0..N-1
old_to_new = {old: new for new, old in enumerate(sorted(valid_idx_set))}
full_dataset.samples = [(p, old_to_new[l]) for p, l in full_dataset.samples]
full_dataset.classes = valid_classes
NUM_CLASSES = len(valid_classes)
print(f'Total images: {len(full_dataset.samples)}, Classes: {NUM_CLASSES}')

# Train / val split
val_n   = int(len(full_dataset) * VAL_SPLIT)
train_n = len(full_dataset) - val_n
train_ds, val_ds = random_split(full_dataset, [train_n, val_n])
train_ds.dataset.transform = train_tf
val_ds.dataset.transform   = val_tf

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# ── 6. Model — EfficientNet-Lite B2 via timm ──────────────────────────────────
model = timm.create_model(
    'efficientnet_lite2',
    pretrained=True,
    num_classes=NUM_CLASSES,
).to(DEVICE)

# Freeze backbone for first 5 epochs (warm up the head only)
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

print(f'Model: efficientnet_lite2 | Classes: {NUM_CLASSES}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── 7. Training loop ──────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
acc_metric = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(DEVICE)

best_val_acc = 0.0
UNFREEZE_EPOCH = 5  # unfreeze full backbone after warmup

for epoch in range(1, EPOCHS + 1):

    # Unfreeze full model after warmup
    if epoch == UNFREEZE_EPOCH + 1:
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=LR / 5)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - UNFREEZE_EPOCH)
        print(f'Epoch {epoch}: full backbone unfrozen')

    # ── Train
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── Validate
    model.eval()
    acc_metric.reset()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            val_loss  += loss.item()
            acc_metric.update(out, labels)

    val_acc = acc_metric.compute().item()
    scheduler.step()

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_loss={train_loss/len(train_loader):.3f} | '
          f'val_loss={val_loss/len(val_loader):.3f} | '
          f'val_acc={val_acc:.3f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pt')
        print(f'  ✓ New best: {val_acc:.3f}')

print(f'\nTraining done. Best val accuracy: {best_val_acc:.3f}')

In [ ]:
# ── 8. Save class index map ───────────────────────────────────────────────────
# Maps class index → class name (e.g. 0 → 'Toyota GR86 ZN8')
class_map = {str(i): name for i, name in enumerate(full_dataset.classes)}
map_path  = f'{OUTPUT_DIR}/class_map.json'
with open(map_path, 'w') as f:
    json.dump(class_map, f, indent=2)
print(f'Class map saved: {map_path}')

In [ ]:
# ── 9. Export to TFLite (for iOS CoreML via ONNX → TFLite pipeline) ──────────
# Alternative: export to ONNX then convert with coremltools for native iOS
!pip install -q onnx onnxruntime

import torch, timm
model.load_state_dict(torch.load('/content/best_model.pt', map_location='cpu'))
model.eval().cpu()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = '/content/vehicle_classifier.onnx'

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}},
    opset_version=17,
)
print('ONNX exported:', onnx_path)

In [ ]:
# ── 10. Convert ONNX → TFLite ─────────────────────────────────────────────────
!pip install -q onnx-tf tensorflow

import onnx
from onnx_tf.backend import prepare
import tensorflow as tf

onnx_model   = onnx.load(onnx_path)
tf_rep       = prepare(onnx_model)
tf_saved_dir = '/content/tf_saved_model'
tf_rep.export_graph(tf_saved_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(tf_saved_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # int8 quantisation
tflite_model = converter.convert()

tflite_path = f'{OUTPUT_DIR}/vehicle_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1e6
print(f'TFLite model saved: {tflite_path} ({size_mb:.1f} MB)')

In [ ]:
# ── 11. Copy everything to Drive ──────────────────────────────────────────────
import shutil
shutil.copy('/content/best_model.pt', f'{OUTPUT_DIR}/best_model.pt')
shutil.copy(onnx_path, f'{OUTPUT_DIR}/vehicle_classifier.onnx')
print('All artifacts saved to Drive:')
for f in Path(OUTPUT_DIR).iterdir():
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## Next steps

After training:
1. Download `vehicle_classifier.tflite` and `class_map.json` from Drive
2. Place them in `mobile/src/modules/vehicle-classifier/model/`
3. Implement the VisionCamera frame processor plugin in Phase 5
   - iOS: wrap TFLite via `TensorFlowLiteTaskVision` in a Swift VisionCamera plugin
   - Android: wrap via `tflite-task-vision` in a Kotlin VisionCamera plugin
4. Replace `VehicleClassifierStub` with the real `VehicleClassifier` in `highway.tsx` / `scan360.tsx`

### Accuracy targets
| Tier    | Min top-1 accuracy |
|---------|-------------------|
| Legendary / Epic | 85% |
| Rare    | 75% |
| Common  | 65% |

If accuracy is below target, options:
- Scrape more images for underperforming classes (check confusion matrix)
- Increase `EPOCHS` to 40
- Try `efficientnet_lite3` (slightly larger, more accurate)
